In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

In [2]:
df = pd.read_csv('../data/eda_ready_dataset.csv')

In [3]:
y = pd.DataFrame({
    "Cath": (df["Cath"] == "CAD").astype(int),
    "LAD":  (df["LAD"]  == "Stenotic").astype(int),
    "LCX":  (df["LCX"]  == "Stenotic").astype(int),
    "RCA":  (df["RCA"]  == "Stenotic").astype(int),
})

In [4]:
X = df.drop(columns=["Cath","LAD","LCX","RCA","Exertional CP"]).copy()


In [5]:
X["Sex"] = X["Sex"].map({"Male":1, "Female":0})
yn = ["Obesity","CRF","CVA","Airway disease","Thyroid Disease","CHF","DLP",
      "Weak Peripheral Pulse","Lung rales","Systolic Murmur","Diastolic Murmur",
      "Dyspnea","Atypical","Nonanginal","LowTH Ang","LVH","Poor R Progression"]
for c in yn:
    X[c] = X[c].map({"Y":1, "N":0})
X["VHD"] = X["VHD"].str.lower().map({"n":0,"mild":1,"moderate":2,"severe":3})
X = pd.get_dummies(X, columns=["BBB","Region RWMA"], dtype=int)

assert X.isnull().sum().sum() == 0, "Mapping mein spelling galat hai"
print(X.shape)
print(y.mean().round(3))
print("Exertional CP" in X.columns)

(303, 60)
Cath    0.713
LAD     0.584
LCX     0.393
RCA     0.376
dtype: float64
False


In [6]:
CV = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
results = []

def evaluate(name, model, X, target):
    s = cross_validate(model, X, y[target], cv=CV,
                       scoring=["roc_auc","f1","recall","accuracy"], n_jobs=-1)
    row = {
        "experiment": name, "target": target, "n_features": X.shape[1],
        "AUC": s["test_roc_auc"].mean(), "AUC_std": s["test_roc_auc"].std(),
        "F1": s["test_f1"].mean(), "Recall": s["test_recall"].mean(),
        "Acc": s["test_accuracy"].mean(),
    }
    results.append(row)
    print(f"{name:20s} {target:5s} AUC={row['AUC']:.3f} (+/-{row['AUC_std']:.3f})  F1={row['F1']:.3f}  Recall={row['Recall']:.3f}  Acc={row['Acc']:.3f}")

In [7]:
for t in ["Cath", "LAD", "LCX", "RCA"]:
    evaluate("Dummy", DummyClassifier(strategy="prior"), X, t)
    evaluate("LR_scaled", make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), X, t)
    evaluate("RF", RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0), X, t)
    print("-" * 70)

Dummy                Cath  AUC=0.500 (+/-0.000)  F1=0.832  Recall=1.000  Acc=0.713
LR_scaled            Cath  AUC=0.907 (+/-0.028)  F1=0.888  Recall=0.881  Acc=0.843
RF                   Cath  AUC=0.930 (+/-0.031)  F1=0.906  Recall=0.957  Acc=0.858
----------------------------------------------------------------------
Dummy                LAD   AUC=0.500 (+/-0.000)  F1=0.737  Recall=1.000  Acc=0.584
LR_scaled            LAD   AUC=0.808 (+/-0.050)  F1=0.767  Recall=0.761  Acc=0.732
RF                   LAD   AUC=0.854 (+/-0.043)  F1=0.827  Recall=0.885  Acc=0.783
----------------------------------------------------------------------
Dummy                LCX   AUC=0.500 (+/-0.000)  F1=0.000  Recall=0.000  Acc=0.607
LR_scaled            LCX   AUC=0.643 (+/-0.054)  F1=0.488  Recall=0.459  Acc=0.623
RF                   LCX   AUC=0.730 (+/-0.062)  F1=0.446  Recall=0.342  Acc=0.671
----------------------------------------------------------------------
Dummy                RCA   AUC=0.500 (+/

In [8]:
res = pd.DataFrame(results)
print(res.pivot_table(index="experiment", columns="target", values="AUC").round(3))

target       Cath    LAD    LCX    RCA
experiment                            
Dummy       0.500  0.500  0.500  0.500
LR_scaled   0.907  0.808  0.643  0.698
RF          0.930  0.854  0.730  0.724


In [9]:
for t in ["Cath", "LAD", "LCX", "RCA"]:
    evaluate("LR_balanced",
             make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")),
             X, t)
    evaluate("RF_balanced",
             RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced", random_state=0),
             X, t)
    print("-" * 70)

LR_balanced          Cath  AUC=0.906 (+/-0.030)  F1=0.876  Recall=0.841  Acc=0.832
RF_balanced          Cath  AUC=0.925 (+/-0.033)  F1=0.905  Recall=0.897  Acc=0.866
----------------------------------------------------------------------
LR_balanced          LAD   AUC=0.806 (+/-0.050)  F1=0.756  Recall=0.727  Acc=0.729
RF_balanced          LAD   AUC=0.852 (+/-0.044)  F1=0.811  Recall=0.827  Acc=0.777
----------------------------------------------------------------------
LR_balanced          LCX   AUC=0.646 (+/-0.057)  F1=0.542  Recall=0.571  Acc=0.621
RF_balanced          LCX   AUC=0.733 (+/-0.065)  F1=0.604  Recall=0.613  Acc=0.685
----------------------------------------------------------------------
LR_balanced          RCA   AUC=0.697 (+/-0.049)  F1=0.562  Recall=0.609  Acc=0.646
RF_balanced          RCA   AUC=0.729 (+/-0.046)  F1=0.570  Recall=0.570  Acc=0.680
----------------------------------------------------------------------


In [10]:
res = pd.DataFrame(results)

print("=== AUC ===")
print(res.pivot_table(index="experiment", columns="target", values="AUC").round(3))

print("\n=== Recall ===")
print(res.pivot_table(index="experiment", columns="target", values="Recall").round(3))

print("\n=== F1 ===")
print(res.pivot_table(index="experiment", columns="target", values="F1").round(3))

=== AUC ===
target        Cath    LAD    LCX    RCA
experiment                             
Dummy        0.500  0.500  0.500  0.500
LR_balanced  0.906  0.806  0.646  0.697
LR_scaled    0.907  0.808  0.643  0.698
RF           0.930  0.854  0.730  0.724
RF_balanced  0.925  0.852  0.733  0.729

=== Recall ===
target        Cath    LAD    LCX    RCA
experiment                             
Dummy        1.000  1.000  0.000  0.000
LR_balanced  0.841  0.727  0.571  0.609
LR_scaled    0.881  0.761  0.459  0.536
RF           0.957  0.885  0.342  0.310
RF_balanced  0.897  0.827  0.613  0.570

=== F1 ===
target        Cath    LAD    LCX    RCA
experiment                             
Dummy        0.832  0.737  0.000  0.000
LR_balanced  0.876  0.756  0.542  0.562
LR_scaled    0.888  0.767  0.488  0.542
RF           0.906  0.827  0.446  0.406
RF_balanced  0.905  0.811  0.604  0.570


In [11]:
X_noWL = X.drop(columns=["Weight", "Length"])

for t in ["Cath", "LAD", "LCX", "RCA"]:
    evaluate("RF_bal_allfeat", RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced", random_state=0), X, t)
    evaluate("RF_bal_noWL",    RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced", random_state=0), X_noWL, t)
    print("-" * 70)

RF_bal_allfeat       Cath  AUC=0.925 (+/-0.033)  F1=0.905  Recall=0.897  Acc=0.866
RF_bal_noWL          Cath  AUC=0.930 (+/-0.029)  F1=0.906  Recall=0.895  Acc=0.868
----------------------------------------------------------------------
RF_bal_allfeat       LAD   AUC=0.852 (+/-0.044)  F1=0.811  Recall=0.827  Acc=0.777
RF_bal_noWL          LAD   AUC=0.853 (+/-0.049)  F1=0.832  Recall=0.848  Acc=0.800
----------------------------------------------------------------------
RF_bal_allfeat       LCX   AUC=0.733 (+/-0.065)  F1=0.604  Recall=0.613  Acc=0.685
RF_bal_noWL          LCX   AUC=0.730 (+/-0.066)  F1=0.607  Recall=0.619  Acc=0.685
----------------------------------------------------------------------
RF_bal_allfeat       RCA   AUC=0.729 (+/-0.046)  F1=0.570  Recall=0.570  Acc=0.680
RF_bal_noWL          RCA   AUC=0.725 (+/-0.041)  F1=0.560  Recall=0.561  Acc=0.671
----------------------------------------------------------------------


In [12]:
X = X.drop(columns=["Weight", "Length"])
print(X.shape)   # (303, 58) aana chahiye

(303, 58)


In [13]:
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

for t in ["Cath", "LAD", "LCX", "RCA"]:
    pos_weight = (y[t] == 0).sum() / (y[t] == 1).sum()

    evaluate("RF_bal", RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced", random_state=0), X, t)
    evaluate("ExtraTrees_bal", ExtraTreesClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced", random_state=0), X, t)
    evaluate("GradBoost", GradientBoostingClassifier(n_estimators=100, max_depth=2, learning_rate=0.05, subsample=0.8, random_state=0), X, t)
    evaluate("XGB_shallow", XGBClassifier(n_estimators=150, max_depth=2, learning_rate=0.05, subsample=0.8,
                                          colsample_bytree=0.8, scale_pos_weight=pos_weight,
                                          eval_metric="logloss", random_state=0, verbosity=0), X, t)
    print("-" * 70)

RF_bal               Cath  AUC=0.930 (+/-0.029)  F1=0.906  Recall=0.895  Acc=0.868
ExtraTrees_bal       Cath  AUC=0.932 (+/-0.028)  F1=0.909  Recall=0.883  Acc=0.873
GradBoost            Cath  AUC=0.923 (+/-0.035)  F1=0.906  Recall=0.927  Acc=0.862
XGB_shallow          Cath  AUC=0.920 (+/-0.038)  F1=0.901  Recall=0.881  Acc=0.862
----------------------------------------------------------------------
RF_bal               LAD   AUC=0.853 (+/-0.049)  F1=0.832  Recall=0.848  Acc=0.800
ExtraTrees_bal       LAD   AUC=0.842 (+/-0.046)  F1=0.826  Recall=0.844  Acc=0.792
GradBoost            LAD   AUC=0.839 (+/-0.053)  F1=0.822  Recall=0.853  Acc=0.786
XGB_shallow          LAD   AUC=0.839 (+/-0.055)  F1=0.796  Recall=0.795  Acc=0.764
----------------------------------------------------------------------
RF_bal               LCX   AUC=0.730 (+/-0.066)  F1=0.607  Recall=0.619  Acc=0.685
ExtraTrees_bal       LCX   AUC=0.701 (+/-0.083)  F1=0.569  Recall=0.610  Acc=0.639
GradBoost            LCX   A

In [14]:
from sklearn.feature_selection import SelectKBest, f_classif

for t in ["Cath", "LAD", "LCX", "RCA"]:
    for k in [10, 20, 30, "all"]:
        sel = SelectKBest(f_classif, k=k) if k != "all" else "passthrough"
        pipe = make_pipeline(sel, RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                                        class_weight="balanced", random_state=0))
        evaluate(f"RF_top{k}", pipe, X, t)
    print("-" * 70)

RF_top10             Cath  AUC=0.921 (+/-0.040)  F1=0.895  Recall=0.860  Acc=0.857
RF_top20             Cath  AUC=0.920 (+/-0.033)  F1=0.896  Recall=0.872  Acc=0.856
RF_top30             Cath  AUC=0.926 (+/-0.034)  F1=0.899  Recall=0.880  Acc=0.859
RF_topall            Cath  AUC=0.930 (+/-0.029)  F1=0.906  Recall=0.895  Acc=0.868
----------------------------------------------------------------------
RF_top10             LAD   AUC=0.839 (+/-0.052)  F1=0.804  Recall=0.793  Acc=0.777
RF_top20             LAD   AUC=0.846 (+/-0.048)  F1=0.809  Recall=0.804  Acc=0.779
RF_top30             LAD   AUC=0.850 (+/-0.049)  F1=0.814  Recall=0.816  Acc=0.783
RF_topall            LAD   AUC=0.853 (+/-0.049)  F1=0.832  Recall=0.848  Acc=0.800
----------------------------------------------------------------------
RF_top10             LCX   AUC=0.701 (+/-0.063)  F1=0.570  Recall=0.613  Acc=0.639
RF_top20             LCX   AUC=0.707 (+/-0.063)  F1=0.596  Recall=0.630  Acc=0.666
RF_top30             LCX   A

In [15]:
def add_features(X, which):
    X2 = X.copy()
    if "ecg" in which:
        X2["ECG_any"] = ((X["St Elevation"]==1)|(X["St Depression"]==1)|
                         (X["Tinversion"]==1)|(X["Q Wave"]==1)).astype(int)
    if "risk" in which:
        X2["Risk_count"] = X[["DM","HTN","DLP","Current Smoker","FH"]].sum(axis=1)
    if "nl" in which:
        X2["Neut_Lymph"] = X["Neut"] / X["Lymph"]
    if "chol" in which:
        X2["Chol_ratio"] = X["LDL"] / X["HDL"]
    return X2

rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced", random_state=0)

for t in ["Cath", "LAD", "LCX", "RCA"]:
    evaluate("RF_base", rf, X, t)
    for name in ["ecg", "risk", "nl", "chol"]:
        evaluate(f"RF_+{name}", rf, add_features(X, [name]), t)
    evaluate("RF_+all4", rf, add_features(X, ["ecg","risk","nl","chol"]), t)
    print("-" * 70)

RF_base              Cath  AUC=0.930 (+/-0.029)  F1=0.906  Recall=0.895  Acc=0.868
RF_+ecg              Cath  AUC=0.934 (+/-0.029)  F1=0.908  Recall=0.904  Acc=0.869
RF_+risk             Cath  AUC=0.930 (+/-0.030)  F1=0.908  Recall=0.903  Acc=0.869
RF_+nl               Cath  AUC=0.928 (+/-0.032)  F1=0.904  Recall=0.895  Acc=0.865
RF_+chol             Cath  AUC=0.928 (+/-0.031)  F1=0.906  Recall=0.898  Acc=0.867
RF_+all4             Cath  AUC=0.930 (+/-0.031)  F1=0.909  Recall=0.908  Acc=0.870
----------------------------------------------------------------------
RF_base              LAD   AUC=0.853 (+/-0.049)  F1=0.832  Recall=0.848  Acc=0.800
RF_+ecg              LAD   AUC=0.855 (+/-0.049)  F1=0.821  Recall=0.846  Acc=0.784
RF_+risk             LAD   AUC=0.855 (+/-0.048)  F1=0.827  Recall=0.846  Acc=0.794
RF_+nl               LAD   AUC=0.851 (+/-0.053)  F1=0.820  Recall=0.831  Acc=0.788
RF_+chol             LAD   AUC=0.854 (+/-0.050)  F1=0.825  Recall=0.844  Acc=0.791
RF_+all4        

In [16]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score

param_dist = {
    "n_estimators": [200, 400, 600],
    "max_depth": [3, 5, 8, None],
    "min_samples_leaf": [1, 3, 5, 8],
    "max_features": ["sqrt", 0.3, 0.5],
}

inner = StratifiedKFold(5, shuffle=True, random_state=1)   # tuning ke liye
outer = StratifiedKFold(5, shuffle=True, random_state=2)   # final score ke liye

for t in ["Cath", "LAD", "LCX", "RCA"]:
    base = RandomForestClassifier(class_weight="balanced", random_state=0)
    search = RandomizedSearchCV(base, param_dist, n_iter=15, scoring="roc_auc",
                                cv=inner, random_state=0, n_jobs=-1)
    nested = cross_val_score(search, X, y[t], cv=outer, scoring="roc_auc")
    print(f"{t:5s} Tuned RF nested AUC = {nested.mean():.3f} (+/-{nested.std():.3f})")

Cath  Tuned RF nested AUC = 0.926 (+/-0.030)
LAD   Tuned RF nested AUC = 0.860 (+/-0.042)
LCX   Tuned RF nested AUC = 0.738 (+/-0.033)
RCA   Tuned RF nested AUC = 0.711 (+/-0.030)


In [17]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import cross_val_predict

cvp = StratifiedKFold(5, shuffle=True, random_state=42)

def rf():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                  class_weight="balanced", random_state=0)

for t in ["Cath", "LAD", "LCX", "RCA"]:
    models = {
        "RF_raw":      rf(),
        "RF_sigmoid":  CalibratedClassifierCV(rf(), method="sigmoid", cv=5),
        "RF_isotonic": CalibratedClassifierCV(rf(), method="isotonic", cv=5),
    }
    for name, m in models.items():
        p = cross_val_predict(m, X, y[t], cv=cvp, method="predict_proba")[:, 1]
        print(f"{name:12s} {t:5s} Brier={brier_score_loss(y[t], p):.3f}  AUC={roc_auc_score(y[t], p):.3f}")
    print("-" * 60)

RF_raw       Cath  Brier=0.119  AUC=0.923
RF_sigmoid   Cath  Brier=0.098  AUC=0.926
RF_isotonic  Cath  Brier=0.098  AUC=0.923
------------------------------------------------------------
RF_raw       LAD   Brier=0.165  AUC=0.853
RF_sigmoid   LAD   Brier=0.152  AUC=0.849
RF_isotonic  LAD   Brier=0.155  AUC=0.841
------------------------------------------------------------
RF_raw       LCX   Brier=0.210  AUC=0.732
RF_sigmoid   LCX   Brier=0.207  AUC=0.723
RF_isotonic  LCX   Brier=0.205  AUC=0.729
------------------------------------------------------------
RF_raw       RCA   Brier=0.208  AUC=0.726
RF_sigmoid   RCA   Brier=0.203  AUC=0.724
RF_isotonic  RCA   Brier=0.204  AUC=0.718
------------------------------------------------------------
